# Ablations — 7 runs off the P1 entrypoint (v2)
T4 GPU, ~70-90 min total. Run ONLY after P1 looks sane. Each run banks before the next starts, so a dead session only loses the run in progress — rerun this notebook and it continues (already-banked results just get overwritten with identical numbers, seed is fixed).

In [ ]:
# cell 1 — environment (rerun if the session dies)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf repo
!git clone -b fix/v2-reproducibility-foundation https://github.com/ruwini01/Sinhala_English_Code_Mixed_Sentiment_Analysis.git repo
%cd /content/repo/ml
!pip install -q peft
!pip uninstall -y -q torchao

import os, shutil
SAVE = "/content/drive/MyDrive/Final_Reporing_Sentiment_Analysis/thesis_v2"
for sub in ("results", "checkpoints", "tokenized"):
    os.makedirs(f"{SAVE}/{sub}", exist_ok=True)

def bank(*paths, sub="results"):
    """Copy artifacts to Drive IMMEDIATELY — sessions die without warning."""
    for p in paths:
        if os.path.isdir(p):
            shutil.copytree(p, f"{SAVE}/{sub}/{os.path.basename(p)}", dirs_exist_ok=True)
        elif os.path.exists(p):
            shutil.copy2(p, f"{SAVE}/{sub}/")
        else:
            print("MISSING (not banked):", p)
    print("banked ->", f"{SAVE}/{sub}:", ", ".join(os.path.basename(p) for p in paths))

In [ ]:
# cell 2 — HF token from Colab Secrets (key icon in left sidebar, name: HF_TOKEN)
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# cell 3 — data: raw csv -> preprocess (splits are LOCKED in the repo)
import hashlib, os
RAW = "data/raw/singlish_mixed_sentiment_complete.csv"
os.makedirs("data/raw", exist_ok=True)

DRIVE_RAW = f"{SAVE}/singlish_mixed_sentiment_complete.csv"
if os.path.exists(DRIVE_RAW):
    shutil.copy2(DRIVE_RAW, RAW)
else:
    from google.colab import files
    files.upload()                      # pick the raw CSV from your PC
    os.replace("singlish_mixed_sentiment_complete.csv", RAW)
    shutil.copy2(RAW, DRIVE_RAW)        # bank the raw file itself

sha = hashlib.sha256(open(RAW, "rb").read()).hexdigest()
assert sha == "5ca2952ebf1087dbc0704beb4c53306bf3e30ab201ff72a60171565a932ba221", f"WRONG RAW FILE — sha256 {sha[:16]}... != v2.1 (see ml/DATA.md)"
print("raw sha256 verified: v2.1")

!python -m src.preprocess.quarantine
!python -m src.preprocess.clean_text
!python -m src.preprocess.language_id
# tokenized cache: restore from Drive if a previous notebook banked it, else build + bank
if all(os.path.exists(f"{SAVE}/tokenized/{s}.pt") for s in ("train", "val", "test")):
    os.makedirs("data/processed/tokenized", exist_ok=True)
    for s in ("train", "val", "test"):
        shutil.copy2(f"{SAVE}/tokenized/{s}.pt", f"data/processed/tokenized/{s}.pt")
    print("tokenized cache restored from Drive")
else:
    !python -m src.preprocess.tokenize_cache
    bank("data/processed/tokenized/train.pt", "data/processed/tokenized/val.pt",
         "data/processed/tokenized/test.pt", sub="tokenized")

In [ ]:
# cell 4 — ablation batch (each result banked before the next run starts)
ablations = [
    ("abl_no_scl",    "--id abl_no_scl --lam 0"),
    ("abl_no_lid",    "--id abl_no_lid --no-lid"),
    ("abl_lora_only", "--id abl_lora_only --lam 0 --no-lid"),
    ("abl_no_lora",   "--id abl_no_lora --no-lora --lr 2e-5"),
    ("abl_rank_4",    "--id abl_rank_4 --rank 4"),
    ("abl_rank_16",   "--id abl_rank_16 --rank 16"),
    ("abl_q_only",    "--id abl_q_only --targets query"),
]
for exp_id, flags in ablations:
    if os.path.exists(f"{SAVE}/results/{exp_id}.json"):
        print(f"SKIP {exp_id} — already banked to Drive")
        continue
    print(f"\n{'='*60}\n{exp_id}\n{'='*60}")
    !python -m src.train.run_lora_scl_lid {flags}
    bank(f"results/{exp_id}.json")